# 📊 JobClarity - Feature Engineering

## Objective

Prepare the cleaned dataset for machine learning by creating numerical features.

### Tasks

- Load cleaned dataset
- Build combined text feature
- Convert text into TF-IDF vectors
- Prepare training and testing data

In [27]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [29]:
import importlib
import src.preprocessing.text_cleaner as tc

importlib.reload(tc)

<module 'src.preprocessing.text_cleaner' from 'd:\\DIVY\\CODING\\PURECODING\\AI-ML\\Week9\\Devops\\JobClarity\\src\\preprocessing\\text_cleaner.py'>

In [30]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_jobs.csv")

In [31]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

In [32]:
for column in text_columns:
    df[column] = df[column].apply(tc.clean_text)

In [34]:
df[text_columns].isna().sum()

title              0
company_profile    0
description        0
requirements       0
benefits           0
dtype: int64

In [35]:
import importlib
import src.features.tfidf_vectorizer as tfidf

importlib.reload(tfidf)

vectorizer = tfidf.create_tfidf_vectorizer()

print(vectorizer)

TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')


In [36]:
X = vectorizer.fit_transform(df["combined_text"])

print(type(X))
print(X.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(17880, 10000)


In [37]:
y = df["fraudulent"]

print(y.shape)
print(y.value_counts())

(17880,)
fraudulent
0    17014
1      866
Name: count, dtype: int64


In [38]:
print("Vocabulary Size:", len(vectorizer.vocabulary_))

Vocabulary Size: 10000


In [39]:
list(vectorizer.vocabulary_.items())[:20]

[('marketing', np.int64(5421)),
 ('intern', np.int64(4680)),
 ('food52', np.int64(3676)),
 ('ve', np.int64(9483)),
 ('created', np.int64(2085)),
 ('groundbreaking', np.int64(3987)),
 ('award', np.int64(821)),
 ('winning', np.int64(9750)),
 ('cooking', np.int64(1992)),
 ('site', np.int64(8158)),
 ('support', np.int64(8692)),
 ('connect', np.int64(1858)),
 ('celebrate', np.int64(1361)),
 ('home', np.int64(4232)),
 ('cooks', np.int64(1993)),
 ('need', np.int64(5836)),
 ('place', np.int64(6562)),
 ('editorial', np.int64(2846)),
 ('business', np.int64(1171)),
 ('engineering', np.int64(3014))]

In [40]:
from sklearn.model_selection import train_test_split

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [42]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (14304, 10000)
X_test  : (3576, 10000)
y_train : (14304,)
y_test  : (3576,)


In [43]:
print("Train Distribution")

print(y_train.value_counts(normalize=True))

print()

print("Test Distribution")

print(y_test.value_counts(normalize=True))

Train Distribution
fraudulent
0    0.951552
1    0.048448
Name: proportion, dtype: float64

Test Distribution
fraudulent
0    0.951622
1    0.048378
Name: proportion, dtype: float64


In [44]:
import joblib
from pathlib import Path

In [45]:
MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(exist_ok=True)

In [46]:
joblib.dump(
    vectorizer,
    MODEL_DIR / "tfidf_vectorizer.pkl"
)

print("✅ TF-IDF Vectorizer Saved Successfully")

✅ TF-IDF Vectorizer Saved Successfully


In [47]:
loaded_vectorizer = joblib.load(
    MODEL_DIR / "tfidf_vectorizer.pkl"
)

print(type(loaded_vectorizer))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [48]:
sample = [
    "python machine learning engineer remote job"
]

sample_vector = loaded_vectorizer.transform(sample)

print(sample_vector.shape)

(1, 10000)
